# Agent-Level Analysis

Uses the aggregated output of `X1_data.ipynb` (`agent_level_data.parquet`) to
produce the paper's agent-level descriptive results (`§What drives belief
revisions?`): 
1. estimated marginal means and planned contrasts for
**plasticity**, **directedness**, and **outgoing influence** across the four
scenarios and two network types,
2. the **variance decomposition / ICC** analysis behind Fig. 3 (opinion leaders and followers).

## Requirements

Mixed-effects models are fit in R (`lme4`/`lmerTest`/`emmeans`/`RLRsim`) via
`rpy2`, matching the statistical machinery described in `Appendix E`.

1. Install R (e.g. from https://cran.r-project.org/bin/macosx/).
2. `Rscript -e 'install.packages(c("nloptr", "lme4", "lmerTest", "emmeans", "pbkrtest", "RLRsim"), repos="https://cran.r-project.org")'`
3. `uv add rpy2` (or `pip install rpy2`).

## Note on clustering structure

For a given scenario, role assignments and adjacency matrices are fixed across
statements: for any `(graph_type, seed)` pair, the network topology and role
assignments are identical regardless of which statement is being discussed.
This creates a non-independent dimension in the data:
1. The same agent-network configuration is reused across statements, so
   observations from the same `(graph_type, seed)` pair but different
   statements are also not independent.

We therefore use **crossed random effects**, `(1 | statement_id) + (1 | graph_type:graph_seed)`
(via `lme4`), rather than clustering on a single grouping factor.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from r_utils import fit_lmer_full
from plot_utils import _configure_fonts, plot_emmeans_grouped_bar, SETTING_MAP, OKABE_ITO

_configure_fonts()


In [ ]:
def save_model_results(results_dict: dict, name: str, save_path: Path) -> None:
    """Save a `fit_lmer_full`/`fit_lmer_with_contrasts` results dict to CSV/JSON."""
    save_path = save_path / name
    save_path.mkdir(parents=True, exist_ok=True)
    for key, df in results_dict.items():
        if isinstance(df, pd.DataFrame):
            df.to_csv(save_path / f"{key}.csv", index=True)
        else:
            with open(save_path / f"{key}.json", "w") as f:
                json.dump(df, f, indent=4)


## Load data

In [ ]:
data_path = Path.cwd().parent / "data" / "outputs" / "aggregated"
result_path = Path.cwd().parent / "data" / "analysis"

agg_path = result_path / "aggregated" / "agent"
cts_path = result_path / "contrasts" / "agent"
agg_path.mkdir(parents=True, exist_ok=True)
cts_path.mkdir(parents=True, exist_ok=True)

dft = pl.read_parquet(data_path / "agent_level_data.parquet").to_pandas()

SETTINGS = ["base_llms", "random_roles", "random_experts", "experts"]
dft["setting"] = pd.Categorical(dft["setting"], categories=SETTINGS)
for col in ["model", "graph_type", "role", "statement_id", "graph_seed", "run_id"]:
    dft[col] = pd.Categorical(dft[col], categories=sorted(dft[col].astype(str).unique()))

dft.head()


## 1. Estimated marginal means (Fig. 2A/B)

Fit `<metric> ~ scenario * graph_type + (1 | statement) + (1 | graph_type:graph_seed)`
for each of the three agent-level outcomes (e.g., plasticity), plus a graph-type-pooled version
(the "marginal" estimates reported alongside the per-network ones in
`Appendix E`).


In [ ]:
COL_PRETTY_NAMES = {
    "influence_out_joint": "Influence",
    "plasticity_tv": "Plasticity",
    "monotonicity": "Directedness",
}

for var_name in COL_PRETTY_NAMES:
    res = fit_lmer_full(
        df=dft,
        specification=f"{var_name} ~ setting * graph_type + (1 | statement_id) + (1 | graph_type:graph_seed)",
        emm_formula="~setting * graph_type",
        reml=True,
    )
    res_pooled = fit_lmer_full(
        df=dft,
        specification=f"{var_name} ~ setting + (1 | statement_id) + (1 | graph_type:graph_seed)",
        emm_formula="~setting",
        reml=True,
    )
    save_model_results(res, f"{var_name}_setting_graph_type", agg_path)
    save_model_results(res_pooled, f"{var_name}_setting", agg_path)
    print(f"Finished fitting model for {var_name} with setting * graph_type")

    plot_emmeans_grouped_bar(
        emm_df=res["emm"],
        setting_reference=dft,
        y_label=COL_PRETTY_NAMES[var_name],
        title="",
        figsize=(4.5, 3),
        show=True,
        out_fig=agg_path / f"{var_name}_grouped_bar.pdf",
    )


## 2. Variance decomposition and ICC for outgoing influence (Fig. 3)

To test whether some agents are persistent "opinion leaders" (LLM agents with 
systematically higher outgoing influence than others), we fit a 
variance-components model with random intercepts for agent identity 
(nested within network), network realization, and statement, then report 
the **intraclass correlation (ICC)** attributable to agent identity.

Significance of the agent-level variance component is tested with an **exact
restricted likelihood-ratio test** (`RLRsim::exactRLRT`), which correctly
handles the boundary problem in testing a variance component against zero
(a naive Wald test is invalid here since variances cannot be negative).


In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri


def fit_lmer_with_icc(df: pd.DataFrame, specification: str, reml: bool = True) -> dict:
    """Fit lmer and extract variance components + ICC for agent_id:graph_id."""
    with (ro.default_converter + pandas2ri.converter).context():
        ro.globalenv["df"] = ro.conversion.py2rpy(df)
        ro.globalenv["spec"] = specification
        ro.globalenv["use_reml"] = reml

        ro.r('''
            library(lmerTest)
            df_model <- lmerTest::lmer(as.formula(spec), data = df, REML = use_reml)

            vc <- as.data.frame(VarCorr(df_model))
            sigma_agent <- vc[vc$grp == "agent_id:graph_id", "vcov"]
            sigma_graph <- vc[vc$grp == "graph_id", "vcov"]
            sigma_stmt  <- vc[vc$grp == "statement_id", "vcov"]
            sigma_resid <- vc[vc$grp == "Residual", "vcov"]
            sigma_total <- sigma_agent + sigma_graph + sigma_stmt + sigma_resid

            # Parametric bootstrap for ICC confidence intervals.
            icc_fn <- function(fit) {
                vc <- as.data.frame(VarCorr(fit))
                sa <- vc[vc$grp == "agent_id:graph_id", "vcov"]
                sg <- vc[vc$grp == "graph_id", "vcov"]
                ss <- vc[vc$grp == "statement_id", "vcov"]
                sr <- vc[vc$grp == "Residual", "vcov"]
                st <- sa + sg + ss + sr
                c(icc_agent = sa/st, icc_graph = sg/st, icc_stmt = ss/st, icc_resid = sr/st)
            }
            boot_result <- bootMer(df_model, FUN = icc_fn, nsim = 1000, type = "parametric", use.u = FALSE)
            boot_ci <- apply(boot_result$t, 2, quantile, probs = c(0.025, 0.975), na.rm = TRUE)

            icc_df <- data.frame(
                icc_agent = sigma_agent / sigma_total, icc_graph = sigma_graph / sigma_total,
                icc_stmt  = sigma_stmt  / sigma_total, icc_resid = sigma_resid / sigma_total,
                var_agent = sigma_agent, var_graph = sigma_graph, var_stmt = sigma_stmt, var_resid = sigma_resid,
                icc_agent_lo = boot_ci[1, "icc_agent"], icc_agent_hi = boot_ci[2, "icc_agent"],
                icc_graph_lo = boot_ci[1, "icc_graph"], icc_graph_hi = boot_ci[2, "icc_graph"],
                icc_stmt_lo  = boot_ci[1, "icc_stmt"],  icc_stmt_hi  = boot_ci[2, "icc_stmt"],
                icc_resid_lo = boot_ci[1, "icc_resid"], icc_resid_hi = boot_ci[2, "icc_resid"]
            )
        ''')

        return {
            "vc": ro.conversion.rpy2py(ro.r("vc")),
            "icc": ro.conversion.rpy2py(ro.r("icc_df")),
        }


def lrt_agent_variance(df: pd.DataFrame) -> dict:
    """Exact restricted LRT for the agent_id:graph_id variance component on outgoing influence.

    Tested on `influence_out_joint` (Eq. 6) - the same metric the ICC above
    decomposes - so the reported p-value and the reported ICC always refer to
    the same quantity.
    """
    with (ro.default_converter + pandas2ri.converter).context():
        ro.globalenv["df"] = ro.conversion.py2rpy(df)

        ro.r('''
            library(lme4)
            library(RLRsim)

            m_full <- lmer(
                influence_out_joint ~ 1 + (1 | graph_id) + (1 | statement_id) + (1 | agent_id:graph_id),
                data = df, REML = TRUE
            )
            m_reduced <- lmer(
                influence_out_joint ~ 1 + (1 | graph_id) + (1 | statement_id),
                data = df, REML = TRUE
            )
            m_alt <- lmer(
                influence_out_joint ~ 1 + (1 | agent_id:graph_id),
                data = df, REML = TRUE
            )

            test <- exactRLRT(m = m_alt, mA = m_full, m0 = m_reduced, nsim = 10000)
            lrt_stat <- as.numeric(test$statistic)
            lrt_p    <- as.numeric(test$p.value)
        ''')

        return {
            "lrt_stat": float(ro.r("lrt_stat")[0]),
            "lrt_p": float(ro.r("lrt_p")[0]),
        }


In [ ]:
icc_results = []
for setting in dft["setting"].unique():
    sub = dft[dft["setting"] == setting].copy()
    res = fit_lmer_with_icc(
        sub,
        specification="influence_out_joint ~ 1 + (1 | graph_id) + (1 | statement_id) + (1 | agent_id:graph_id)",
    )
    lrt_res = lrt_agent_variance(sub)

    icc_row = res["icc"].iloc[0].to_dict()
    icc_row["setting"] = setting
    icc_row["lrt_stat"] = lrt_res["lrt_stat"]
    icc_row["lrt_p"] = lrt_res["lrt_p"]
    icc_results.append(icc_row)

icc_df = pd.DataFrame(icc_results)
icc_df.to_csv(agg_path / "influence_out_icc.csv", index=False)
icc_df[["setting", "icc_agent", "icc_agent_lo", "icc_agent_hi", "icc_graph", "icc_graph_lo", "icc_graph_hi", "lrt_p"]]


### LaTeX table

In [ ]:
scenario_order = ["base_llms", "random_roles", "random_experts", "experts"]
scenario_labels = {
    "base_llms": "I. Baseline",
    "random_roles": "II. Random Roles",
    "random_experts": "III. Random specialists",
    "experts": "IV. Specialists (matched)",
}

tab = (
    icc_df.copy()
    .assign(
        Scenario=lambda d: d["setting"].map(scenario_labels),
        rlrt_p_fmt=lambda d: d["lrt_p"].map(lambda x: "<0.001" if pd.notna(x) and x < 0.001 else f"{x:.2f}"),
    )
    .set_index("setting")
    .reindex(scenario_order)
    .reset_index()
)

pct_cols = ["icc_agent", "icc_agent_lo", "icc_agent_hi", "icc_graph", "icc_graph_lo", "icc_graph_hi"]
tab[pct_cols] = tab[pct_cols] * 100

out = pd.DataFrame({
    "Scenario": tab["Scenario"],
    ("Agent ICC (95\\% CI)", "ICC"): tab["icc_agent"].round(3),
    ("Agent ICC (95\\% CI)", "Lower"): tab["icc_agent_lo"].round(3),
    ("Agent ICC (95\\% CI)", "Upper"): tab["icc_agent_hi"].round(3),
    ("Network ICC (95\\% CI)", "ICC"): tab["icc_graph"].round(3),
    ("Network ICC (95\\% CI)", "Lower"): tab["icc_graph_lo"].round(3),
    ("Network ICC (95\\% CI)", "Upper"): tab["icc_graph_hi"].round(3),
    ("", "RLRT $p$"): tab["rlrt_p_fmt"],
})
out.columns = pd.MultiIndex.from_tuples([(" ", "Scenario")] + list(out.columns[1:]))

print(out.to_latex(
    index=False, escape=False, multicolumn=True, multicolumn_format="c",
    column_format="lccc|ccc|l", float_format="{:.2f}".format,
    caption="Variance decomposition for influence with agent and network ICCs.",
    label="tab:icc-influence",
))

### Plot (Fig. 3)

In [ ]:
def plot_icc_grouped_bar(
    icc_df: pd.DataFrame,
    setting_reference: pd.DataFrame | None = None,
    out_fig: Path | None = None,
    y_label: str = "Intraclass correlation (ICC, %)",
    figsize: tuple[float, float] = (4.5, 3),
):
    """Grouped ICC bars (agent vs. network component) with asymmetric CIs."""
    from matplotlib.ticker import PercentFormatter

    _configure_fonts()

    setting_order = [
        s for s in (setting_reference["setting"].astype(str).unique() if setting_reference is not None
                    else ["base_llms", "random_roles", "random_experts", "experts"])
        if s in set(icc_df["setting"].astype(str))
    ]
    component_order = ["icc_agent", "icc_graph"]
    component_labels = {"icc_agent": "Agent", "icc_graph": "Network"}

    plot_df = icc_df.copy()
    plot_df["setting"] = plot_df["setting"].astype(str)
    pivot_mean = plot_df.set_index("setting")[component_order].reindex(setting_order).astype(float)
    pivot_lcl = (
        plot_df.set_index("setting")[["icc_agent_lo", "icc_graph_lo"]]
        .rename(columns={"icc_agent_lo": "icc_agent", "icc_graph_lo": "icc_graph"})
        .reindex(setting_order).astype(float)
    )
    pivot_ucl = (
        plot_df.set_index("setting")[["icc_agent_hi", "icc_graph_hi"]]
        .rename(columns={"icc_agent_hi": "icc_agent", "icc_graph_hi": "icc_graph"})
        .reindex(setting_order).astype(float)
    )

    x = np.arange(len(setting_order))
    width = 0.4
    colors = [OKABE_ITO["sky"], OKABE_ITO["vermilion"]]
    ci_color = OKABE_ITO["black"]

    fig, ax = plt.subplots(figsize=figsize)
    for j, comp in enumerate(component_order):
        y = pivot_mean[comp].to_numpy()
        yerr = np.vstack([y - pivot_lcl[comp].to_numpy(), pivot_ucl[comp].to_numpy() - y])
        ax.bar(
            x + (j - 0.5) * width, y, width=width - 0.05, label=component_labels[comp],
            color=colors[j], edgecolor=ci_color, linewidth=1.5,
            yerr=yerr, capsize=4, error_kw={"elinewidth": 1.3, "ecolor": ci_color}, alpha=0.95,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([SETTING_MAP.get(s, s) for s in setting_order], rotation=15, ha="center")
    ax.set_xlabel("Setting")
    ax.set_ylabel(y_label)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.legend(frameon=False, loc="upper left")
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=1)
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(0.05, float(np.nanmax(pivot_ucl.to_numpy())) * 1.15))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.tick_params(width=1.3, length=5)
    plt.tight_layout()

    if out_fig is not None:
        fig.savefig(out_fig, bbox_inches="tight", format="pdf")
        print(f"Saved figure to: {out_fig}")
    plt.show()
    return fig, ax


plot_icc_grouped_bar(
    icc_df=icc_df,
    setting_reference=dft,
    out_fig=agg_path / "influence_out_icc_agent_graph_grouped_bar.pdf",
)


## 3. Contrast analysis (Fig. "contrast forest", Appendix Table "marginal-contrasts")

Four planned, single-degree-of-freedom contrasts on the four scenarios
(`Appendix E.3`): the **role effect** (II vs. I), **specialization effect**
(III vs. II), **role-specialization alignment** (IV vs. III), and
**composition effect** (`avg`(III, IV) vs. `avg`(I, II)). Each is reported as a
standardized Cohen's *d*, both marginally (pooled across network types) and
conditionally within each network type, plus their interaction (whether the
effect differs by network type).

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from typing import Any


def fit_lmer_with_contrasts(
    df: pd.DataFrame,
    outcome: str,
    spec_extra: str = "* graph_type",
    random_effects: str = "(1 | statement_id) + (1 | run_id)",
    reml: bool = True,
    bootstrap: bool = False,
    n_boot: int = 2000,
    parallel: str = "multicore",
    ncpus: int = 4,
    seed: int = 723522,
) -> dict[str, Any]:
    """
    Fit a mixed-effects model with `setting` as a fixed effect and compute the four
    planned contrasts, each with a total-SD Cohen's d.

    Contrasts:
        - role_eff:      random_roles - base_llms       (role effect, model constant)
        - model_eff:     random_experts - random_roles  (model effect, role constant)
        - alignment:     experts - random_experts        (role-model alignment effect)
        - dissociation:  avg(heterogeneous) - avg(homogeneous) # composition

    Effect size
    -----------
    Cohen's d is standardized by the TOTAL SD, sigma_total = sqrt(var_statement +
    var_run + var_resid) - the interpretively correct denominator for a
    within-statement/run contrast (Westfall, Kenny & Judd 2014).

    CI construction
    ---------------
    - bootstrap=True  (reported): parametric bootstrap via lme4::bootMer. Each
      replicate refits the model and recomputes BOTH the contrast estimate AND
      sigma_total, so the d CI reflects sampling uncertainty in the numerator
      *and* the denominator (statement_id has few groups, so a fixed-denominator
      rescaling would understate the true uncertainty).
    - bootstrap=False (fast, for iteration): d CI = raw contrast CI / sigma_total,
      i.e. sigma_total treated as a known constant. Point estimate is identical;
      only the interval is approximate (optimistic).

    Returns
    -------
    dict with: emm, contrasts_{marginal,within,interaction}, fit_stats, boot_diag.
    """
    specification = f"{outcome} ~ setting {spec_extra} + {random_effects}"
    if "graph_type" not in spec_extra:
        raise ValueError("emmeans grids assume graph_type is in spec_extra")

    with (ro.default_converter + pandas2ri.converter).context():
        ro.globalenv["df"] = ro.conversion.py2rpy(df)
        ro.globalenv["spec"] = specification
        ro.globalenv["use_reml"] = reml
        ro.globalenv["do_boot"] = bootstrap
        ro.globalenv["n_boot"] = n_boot
        ro.globalenv["boot_seed"] = seed
        ro.globalenv["boot_parallel"] = parallel
        ro.globalenv["boot_ncpus"] = ncpus

        ro.r('''
            library(lmerTest); library(emmeans); library(lme4)
            emm_options(pbkrtest.limit = 100000, lmerTest.limit = 100000)

            df$setting <- factor(df$setting, levels = c("base_llms", "random_roles", "random_experts", "experts"))
            df_model <- lmerTest::lmer(as.formula(spec), data = df, REML = use_reml)

            contr_list <- list(
                role_eff     = c(-1, 1, 0, 0),
                model_eff    = c(0, -1, 1, 0),
                alignment    = c(0, 0, -1, 1),
                dissociation = c(-0.5, -0.5, 0.5, 0.5)
            )

            # NOTE: sum(vcov) is valid ONLY for intercept-only random effects.
            total_sd <- function(fit) sqrt(sum(as.data.frame(VarCorr(fit))$vcov))
            sigma_total <- total_sd(df_model)

            emm_marginal <- emmeans(df_model, ~ setting, lmer.df = "satterthwaite")
            raw_marginal <- contrast(emm_marginal, contr_list, adjust = "none")
            contrasts_marginal <- as.data.frame(raw_marginal)
            ci_marginal <- as.data.frame(confint(raw_marginal))
            contrasts_marginal$lower.CL <- ci_marginal$lower.CL
            contrasts_marginal$upper.CL <- ci_marginal$upper.CL

            emm_within <- emmeans(df_model, ~ setting | graph_type, lmer.df = "satterthwaite")
            within_obj <- contrast(emm_within, contr_list, adjust = "none")
            contrasts_within <- as.data.frame(within_obj)
            ci_within <- as.data.frame(confint(within_obj))
            contrasts_within$lower.CL <- ci_within$lower.CL
            contrasts_within$upper.CL <- ci_within$upper.CL

            interaction_obj <- pairs(within_obj, by = "contrast", adjust = "none")
            contrasts_interaction <- as.data.frame(interaction_obj)
            ci_interaction <- as.data.frame(confint(interaction_obj))
            contrasts_interaction$lower.CL <- ci_interaction$lower.CL
            contrasts_interaction$upper.CL <- ci_interaction$upper.CL

            n_m <- nrow(contrasts_marginal); n_w <- nrow(contrasts_within); n_i <- nrow(contrasts_interaction)

            contrasts_marginal$cohens_d_total    <- contrasts_marginal$estimate    / sigma_total
            contrasts_within$cohens_d_total      <- contrasts_within$estimate      / sigma_total
            contrasts_interaction$cohens_d_total <- contrasts_interaction$estimate / sigma_total

            boot_diag <- data.frame(n_boot = 0L, n_failed = NA_integer_, frac_failed = NA_real_)

            if (isTRUE(do_boot)) {
                d_total_stat <- function(fit) {
                    em <- emmeans(fit, ~ setting)
                    ew <- emmeans(fit, ~ setting | graph_type)
                    est_m <- as.data.frame(contrast(em, contr_list, adjust = "none"))$estimate
                    wob   <- contrast(ew, contr_list, adjust = "none")
                    est_w <- as.data.frame(wob)$estimate
                    est_i <- as.data.frame(pairs(wob, by = "contrast", adjust = "none"))$estimate
                    c(est_m, est_w, est_i) / total_sd(fit)
                }
                boot_model <- update(df_model, control = lmerControl(calc.derivs = FALSE))
                bb <- lme4::bootMer(boot_model, d_total_stat, nsim = n_boot, seed = boot_seed,
                                     type = "parametric", parallel = boot_parallel, ncpus = boot_ncpus, use.u = FALSE)

                finite_mask <- apply(bb$t, 1, function(row) all(is.finite(row)))
                n_failed <- sum(!finite_mask)
                bt_clean <- bb$t[finite_mask, , drop = FALSE]
                ci_boot  <- t(apply(bt_clean, 2, quantile, probs = c(.025, .975)))

                idx_m <- 1:n_m; idx_w <- (n_m + 1):(n_m + n_w); idx_i <- (n_m + n_w + 1):(n_m + n_w + n_i)
                contrasts_marginal$cohens_d_total_lower    <- ci_boot[idx_m, 1]
                contrasts_marginal$cohens_d_total_upper    <- ci_boot[idx_m, 2]
                contrasts_within$cohens_d_total_lower      <- ci_boot[idx_w, 1]
                contrasts_within$cohens_d_total_upper      <- ci_boot[idx_w, 2]
                contrasts_interaction$cohens_d_total_lower <- ci_boot[idx_i, 1]
                contrasts_interaction$cohens_d_total_upper <- ci_boot[idx_i, 2]
                boot_diag <- data.frame(n_boot = as.integer(n_boot), n_failed = as.integer(n_failed), frac_failed = n_failed / n_boot)
            } else {
                contrasts_marginal$cohens_d_total_lower    <- contrasts_marginal$lower.CL    / sigma_total
                contrasts_marginal$cohens_d_total_upper    <- contrasts_marginal$upper.CL    / sigma_total
                contrasts_within$cohens_d_total_lower      <- contrasts_within$lower.CL      / sigma_total
                contrasts_within$cohens_d_total_upper      <- contrasts_within$upper.CL      / sigma_total
                contrasts_interaction$cohens_d_total_lower <- contrasts_interaction$lower.CL / sigma_total
                contrasts_interaction$cohens_d_total_upper <- contrasts_interaction$upper.CL / sigma_total
            }

            emm_full <- as.data.frame(emmeans(df_model, ~ setting * graph_type, lmer.df = "satterthwaite"))
            fit_stats <- data.frame(AIC = AIC(df_model), BIC = BIC(df_model), logLik = as.numeric(logLik(df_model)), sigma_total = sigma_total)
        ''')

        return {
            "emm": ro.conversion.rpy2py(ro.r("emm_full")),
            "contrasts_marginal": ro.conversion.rpy2py(ro.r("contrasts_marginal")),
            "contrasts_within": ro.conversion.rpy2py(ro.r("contrasts_within")),
            "contrasts_interaction": ro.conversion.rpy2py(ro.r("contrasts_interaction")),
            "fit_stats": ro.conversion.rpy2py(ro.r("fit_stats")),
            "boot_diag": ro.conversion.rpy2py(ro.r("boot_diag")),
        }


In [ ]:
def build_contrast_summary(col_list: list[str], results_dict: dict) -> pd.DataFrame:
    rows = []
    for col in col_list:
        out = results_dict[col]
        for ctype in ("marginal", "within", "interaction"):
            tbl = out.get(f"contrasts_{ctype}")
            if tbl is None:
                continue
            for _, row in tbl.iterrows():
                p = row.get("p.value", pd.NA)
                rows.append({
                    "outcome": col, "contrast_type": ctype, "graph_type": row.get("graph_type", pd.NA),
                    "contrast": row["contrast"], "estimate": row.get("estimate", pd.NA),
                    "SE": row.get("SE", pd.NA), "t": row.get("t.ratio", pd.NA), "df": row.get("df", pd.NA),
                    "p": p, "lower": row.get("lower.CL", pd.NA), "upper": row.get("upper.CL", pd.NA),
                    "cohens_d": row.get("cohens_d_total", pd.NA),
                    "cohens_d_lower": row.get("cohens_d_total_lower", pd.NA),
                    "cohens_d_upper": row.get("cohens_d_total_upper", pd.NA),
                    "sig": (pd.notna(p) and p < 0.05),
                })
    return pd.DataFrame(rows)


col_list = ["influence_out_joint", "plasticity_tv", "monotonicity"]

results_dict = {col: fit_lmer_with_contrasts(dft, outcome=col, bootstrap=False) for col in col_list}
summary = build_contrast_summary(col_list, results_dict)
summary.to_csv(cts_path / "contrast_summary.csv", index=False)
summary.head()


In [ ]:
from matplotlib.lines import Line2D
from plot_utils import GRAPH_COLOR


def plot_contrast_forest(summary, contrasts_order, outcomes_order, graph_types_order=("erdos-renyi", "watts-strogatz"), figsize=(11, 3)):
    """Forest plot: one panel per contrast, one row per outcome, 3 estimates (marginal, ER, WS) each."""
    contrast_labels = {
        "role_eff": "Role Effect", "model_eff": "Model Effect",
        "alignment": "Role Alignment Effect", "dissociation": "Composition Effect",
    }
    triplet_specs = [
        ("marginal", None, "Marginal (pooled)", OKABE_ITO["blue"], 0.0),
        ("within", graph_types_order[0], "Erdos-Renyi", GRAPH_COLOR[graph_types_order[0]], -0.16),
        ("within", graph_types_order[1], "Watts-Strogatz", GRAPH_COLOR[graph_types_order[1]], 0.16),
    ]

    fig, axes = plt.subplots(1, len(contrasts_order), figsize=figsize, sharey=True, sharex=True, squeeze=False)
    axes = axes[0]
    y_positions = {o: i for i, o in enumerate(outcomes_order)}

    for ax, contrast_name in zip(axes, contrasts_order):
        sub_contrast = summary[summary["contrast"] == contrast_name]
        for y in range(len(outcomes_order)):
            ax.axhline(y, color="0.88", linewidth=0.7, alpha=0.7, zorder=0)

        for outcome in outcomes_order:
            if outcome not in y_positions:
                continue
            y_base = y_positions[outcome]
            for ctype, gtype, _, color, y_offset in triplet_specs:
                mask = (sub_contrast["contrast_type"] == ctype) & (sub_contrast["outcome"] == outcome)
                mask &= sub_contrast["graph_type"].isna() if gtype is None else sub_contrast["graph_type"] == gtype
                sub = sub_contrast[mask]
                if sub.empty:
                    continue
                row = sub.iloc[0]
                is_sig = bool(row["sig"])
                ax.errorbar(
                    row["cohens_d"], y_base + y_offset,
                    xerr=[[row["cohens_d"] - row["cohens_d_lower"]], [row["cohens_d_upper"] - row["cohens_d"]]],
                    fmt="o", color=color, markerfacecolor=color if is_sig else "white", markeredgecolor=color,
                    capsize=4, markersize=4, alpha=1.0 if is_sig else 0.45, linewidth=1.4,
                )

        ax.axvline(0, color="k", linestyle="--", alpha=0.35, linewidth=1)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(contrast_labels.get(contrast_name, contrast_name), fontsize=12)
        ax.set_xlabel("Cohen's d", fontsize=12)

    axes[0].set_yticks(range(len(outcomes_order)))
    axes[0].set_yticklabels([COL_PRETTY_NAMES.get(o, o) for o in outcomes_order], fontsize=12)

    legend_handles = [
        Line2D([0], [0], marker="o", color=c, markerfacecolor=c, markeredgecolor=c, linestyle="None", markersize=6, label=lbl)
        for _, _, lbl, c, _ in triplet_specs
    ]
    fig.legend(handles=legend_handles, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


fig = plot_contrast_forest(
    summary,
    contrasts_order=["role_eff", "model_eff", "alignment"],
    outcomes_order=["influence_out_joint", "plasticity_tv", "monotonicity"],
    figsize=(11, 3),
)
fig.savefig(cts_path / "contrast_forest_triplet_panels.pdf", dpi=300, bbox_inches="tight")